# ABLATION A — DenseNet-121 + CBAM (Classification)

**Ablation Question:**
How much of the proposed model's gain comes from CBAM alone, independent of the triplet metric learning training paradigm?
 
**Details:**
* **Architecture:** DenseNet-121 + CBAM (`baseline=False`, `output_dim=2`)
* **Training:** CrossEntropyLoss, Adam (Identical to baseline training). No backbone freezing, full fine-tuning from epoch 1.
* **Evaluation:** Standard Classification Softmax (Identical to baseline)
 
**Comparisons:**
* **Key differences from baseline:** CBAM modules active (`baseline=False`)
* **Key differences from proposed:** Classification loss, no triplet metric learning, no shared-weight comparison during training

In [1]:
import os, sys, json, random, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from PIL import Image

REPO_ROOT = os.path.abspath(os.path.join(os.path.abspath(os.getcwd()), '..'))
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from models.feature_extractor import DenseNetFeatureExtractor
from utils.model_evaluation   import compute_metrics
from dataloader.tDCBAM_trainloader import get_transforms

/home/lawrence/workspace/thesis/thesis/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


### STEP 1 - REPRODUCIBILITY

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f" > [Seed] {seed}")

seed_everything(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" > [Device] {DEVICE}" +
      (f"  ({torch.cuda.get_device_name()})" if torch.cuda.is_available() else ""))

 > [Seed] 42
 > [Device] cuda  (NVIDIA GeForce RTX 5080)


### STEP 2 — CONFIGURATION

In [ ]:
DATASETS = [
    {'dataset': 'cedar',         'name': 'CEDAR'},
    {'dataset': 'bhsig_bengali', 'name': 'BHSig-Bengali'},
    {'dataset': 'bhsig_hindi',   'name': 'BHSig-Hindi'}
]

SPLIT_DIR      = os.path.join(REPO_ROOT, 'data', 'ratio_splits')
CHECKPOINT_DIR = os.path.join(REPO_ROOT, 'checkpoints', 'ablation_splits')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SPLIT_RATIOS = ['60_20_20', '70_15_15']
IMG_SIZE    = 224
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE)
NUM_WORKERS = 4

EPOCHS              = 100
BATCH_SIZE          = 30
LR                  = 1e-3
MOMENTUM            = 0.99
EARLY_STOP_PATIENCE = 10

print(f" > [Ablation A] DenseNet-121 + CBAM — Classification")
print(f" > [Config] Epochs: {EPOCHS} | LR: {LR} | beta1: {MOMENTUM} | Batch: {BATCH_SIZE}")
print(f" > [Config] CBAM: ACTIVE | L2 Norm: OFF | Loss: CrossEntropyLoss")
print(f" > [Config] Targets: {[d['name'] for d in DATASETS]}")

 > [Ablation A] DenseNet-121 + CBAM — Classification
 > [Config] Epochs: 100 | LR: 0.001 | beta1: 0.99 | Batch: 30
 > [Config] CBAM: ACTIVE | L2 Norm: OFF | Loss: CrossEntropyLoss
 > [Config] Targets: ['CEDAR', 'BHSig-Bengali', 'BHSig-Hindi']


### STEP 3 — TRANSFORMS

In [4]:
train_transform = get_transforms(mode='train', input_shape=INPUT_SHAPE)
val_transform   = get_transforms(mode='val',   input_shape=INPUT_SHAPE)

print(" > [Transforms] train_transform: augmentation ON")
print(" > [Transforms] val_transform  : augmentation OFF")

 > [Transforms] train_transform: augmentation ON
 > [Transforms] val_transform  : augmentation OFF


### STEP 4 - DATASETS

In [5]:
class SplitDataset(Dataset):
    """
    Binary classification dataset for Ablation A.
    Labels: 0 = Genuine, 1 = Forged
    """
    def __init__(self, user_dict, transform=None, silent=False):
        self.samples   = []
        self.transform = transform

        for uid, data in user_dict.items():
            gen_key  = next((k for k in data if k.lower() in ('genuine', 'gen')), None)
            forg_key = next((k for k in data if k.lower() in ('forged', 'forgeries', 'forg')), None)
            gen_paths  = data.get(gen_key,  []) if gen_key  else []
            forg_paths = data.get(forg_key, []) if forg_key else []

            for path in gen_paths:
                self.samples.append((path, 0))
            for path in forg_paths:
                self.samples.append((path, 1))

        if not silent:
            n_gen  = sum(1 for _, l in self.samples if l == 0)
            n_forg = sum(1 for _, l in self.samples if l == 1)
            print(f"   SplitDataset: {len(self.samples)} samples "
                  f"({n_gen} genuine + {n_forg} forged) | {len(user_dict)} writers")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except Exception:
            img = torch.zeros(3, IMG_SIZE, IMG_SIZE)
        return img, label

### STEP 5 — TRAINING & EVALUATION UTILITIES

In [6]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate_model(model, loader, criterion=None, device=None, is_val=False, silent=False):
    """
    Handles both validation tracking and final testing.
    Uses Standard Classification Softmax.
    """
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_scores = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            
            if criterion:
                loss = criterion(outputs, labels)
                total_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)[:, 0]   # P(Genuine)
            inverted_labels = (1 - labels).cpu().numpy().tolist()

            all_scores.extend(probs.cpu().numpy().tolist())
            all_labels.extend(inverted_labels)

    # We do not need curve data since we aren't plotting anything
    metrics = compute_metrics(all_labels, all_scores, return_curve_data=False)
    
    if not is_val and not silent:
        print(f"\n{'='*10} FINAL TEST RESULTS {'='*10}")
        for k, fmt in [('eer', ':.2%'), ('auc', ':.4f'), ('threshold', ':.4f'),
                       ('accuracy', ':.2%'), ('precision', ':.2%'),
                       ('recall', ':.2%'), ('f1', ':.2%')]:
            print(f"  {k.upper():<13}: {metrics.get(k, 0):{fmt[1:]}}")
        print("=" * 38)

    if criterion:
        return total_loss / len(loader), metrics
    return metrics


def run_training(train_loader, val_loader, device, epochs, lr, momentum, weight_decay, dataset_name):
    print(f"\n   {'─'*60}")
    print(f"   ABLATION A — DenseNet-121 + CBAM | {dataset_name}")
    print(f"   Epochs: {epochs} max | LR: {lr} | beta1: {momentum} | Batch: {BATCH_SIZE}")
    print(f"   CBAM: ACTIVE | L2 Norm: OFF | Loss: CrossEntropyLoss")
    print(f"   {'─'*60}")

    model = DenseNetFeatureExtractor(
        backbone_name='densenet121', output_dim=2, pretrained=True,
        baseline=False, normalize=False
    ).to(device)

    n_total = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Params: {n_train:,} / {n_total:,} trainable")

    criterion = nn.CrossEntropyLoss()
    scaler    = torch.amp.GradScaler('cuda')

    optimizer = optim.Adam(model.parameters(), lr=lr, betas=(momentum, 0.999), weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)

    best_eer       = float('inf')
    best_acc       = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    trigger        = 0

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_metrics = evaluate_model(model, val_loader, criterion, device, is_val=True)
        
        val_eer = val_metrics['eer']
        val_acc = val_metrics['accuracy']

        print(f"   Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Train Acc: {train_acc:.2%} | Val EER: {val_eer:.2%} | Val Acc: {val_acc:.2%}")

        scheduler.step(val_eer)

        improved = (val_eer < best_eer or (val_eer == best_eer and val_acc > best_acc))
        if improved:
            best_eer, best_acc = val_eer, val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f"   >>> Best weights updated in RAM (Val EER: {val_eer:.2%})")
            trigger = 0
        else:
            trigger += 1
            if trigger >= EARLY_STOP_PATIENCE:
                print(f"   >>> Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_model_wts)
    return model

### STEP 6 — RUN ALL DATASETS AND SPLITS

In [7]:
for ds_ in DATASETS:
    DATASET      = ds_['dataset']
    DATASET_NAME = ds_['name']
    
    print(f"\n\n{'='*100}")
    print(f"{'STARTING DATASET: ' + DATASET_NAME:^100}")
    print(f"{'='*100}")
    
    all_results = {}

    for ratio in SPLIT_RATIOS:
        split_file  = os.path.join(SPLIT_DIR, f"{DATASET}_split_{ratio}.json")
        split_label = ratio.replace('_', ':')

        if not os.path.exists(split_file):
            print(f"  SKIPPED: split file not found ({split_file})")
            continue

        with open(split_file) as f:
            split_data = json.load(f)

        train_dict = split_data['train']
        val_dict   = split_data['val']
        test_dict  = split_data['test']

        # Writer-disjoint integrity check
        assert not (set(train_dict) & set(val_dict)),  "DATA LEAK: train/val"
        assert not (set(train_dict) & set(test_dict)), "DATA LEAK: train/test"
        assert not (set(val_dict)   & set(test_dict)), "DATA LEAK: val/test"

        print(f"  Writers — Train: {len(train_dict)} | Val: {len(val_dict)} | Test: {len(test_dict)}")
        
        train_dataset = SplitDataset(train_dict, transform=train_transform)
        val_dataset   = SplitDataset(val_dict,   transform=val_transform)
        test_dataset  = SplitDataset(test_dict,  transform=val_transform)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
        val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
        test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

        seed_everything(42)
        t0 = time.time()

        trained_model = run_training(
            train_loader=train_loader, val_loader=val_loader, device=DEVICE,
            epochs=EPOCHS, lr=LR, momentum=MOMENTUM, weight_decay=0.0, dataset_name=DATASET_NAME
        )
        t_train = time.time() - t0

        print("\n   Using best epoch weights for final test evaluation")
        final_metrics = evaluate_model(trained_model, test_loader, device=DEVICE, silent=False)

        key = f"{DATASET_NAME} ({split_label})"
        all_results[key] = {
            'dataset':            DATASET_NAME,
            'split':              split_label,
            'ablation':           'A — CBAM only (classification)',
            'train_users':        len(train_dict),
            'val_users':          len(val_dict),
            'test_users':         len(test_dict),
            'eer':                float(final_metrics['eer']),
            'accuracy':           float(final_metrics['accuracy']),
            'auc':                float(final_metrics['auc']),
            'precision':          float(final_metrics.get('precision', 0)),
            'recall':             float(final_metrics.get('recall',    0)),
            'f1':                 float(final_metrics.get('f1',        0)),
            'train_time_seconds': round(t_train, 2),
        }

    # ── Print Summary Table for Current Dataset ───────────────────────────────────
    W = 100
    print(f"\n{'='*W}")
    print(f"{'ABLATION A — DenseNet-121 + CBAM (Classification) | ' + DATASET_NAME:^{W}}")
    print(f"{'='*W}")
    print(f"{'Split':<10} {'Train':<8} {'Val':<8} {'Test':<8} "
          f"{'EER':>8} {'Accuracy':>10} {'AUC':>8} {'F1':>8} {'Time(s)':>10}")
    print(f"{'-'*W}")

    for key, res in all_results.items():
        print(f"{res['split']:<10} {res['train_users']:<8} "
              f"{res['val_users']:<8} {res['test_users']:<8} "
              f"{res['eer']:>8.4f} {res['accuracy']:>10.4f} "
              f"{res['auc']:>8.4f} {res['f1']:>8.4f} "
              f"{res['train_time_seconds']:>10.2f}")
    print(f"{'='*W}")

    # Save JSON explicitly for this dataset
    results_path = os.path.join(CHECKPOINT_DIR, f'ablation_A_{DATASET}_results.json')
    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"\n > Results saved → {results_path}\n")

print(f"\n{'='*100}")
print(f"{'ALL DATASETS COMPLETED SUCCESSFULLY':^100}")
print(f"{'='*100}")



                                      STARTING DATASET: CEDAR                                       
  Writers — Train: 33 | Val: 11 | Test: 11
   SplitDataset: 1584 samples (792 genuine + 792 forged) | 33 writers
   SplitDataset: 528 samples (264 genuine + 264 forged) | 11 writers
   SplitDataset: 528 samples (264 genuine + 264 forged) | 11 writers
 > [Seed] 42

   ────────────────────────────────────────────────────────────
   ABLATION A — DenseNet-121 + CBAM | CEDAR
   Epochs: 100 max | LR: 0.001 | beta1: 0.99 | Batch: 30
   CBAM: ACTIVE | L2 Norm: OFF | Loss: CrossEntropyLoss
   ────────────────────────────────────────────────────────────
   Params: 7,564,554 / 7,564,554 trainable


Training:   0%|          | 0/52 [00:00<?, ?it/s]

   Epoch 01/100 | Train Loss: 0.7284 | Val Loss: 1.4238 | Train Acc: 63.01% | Val EER: 37.88% | Val Acc: 62.31%
   >>> Best weights updated in RAM (Val EER: 37.88%)


Training:   0%|          | 0/52 [00:00<?, ?it/s]

   Epoch 02/100 | Train Loss: 0.6084 | Val Loss: 1.3150 | Train Acc: 69.42% | Val EER: 30.68% | Val Acc: 69.32%
   >>> Best weights updated in RAM (Val EER: 30.68%)


...

   Epoch 23/100 | Train Loss: 0.0717 | Val Loss: 0.5998 | Train Acc: 97.36% | Val EER: 13.44% | Val Acc: 86.52%


Training:   0%|          | 0/172 [00:00<?, ?it/s]

   Epoch 24/100 | Train Loss: 0.0640 | Val Loss: 0.5487 | Train Acc: 97.69% | Val EER: 12.40% | Val Acc: 87.67%


Training:   0%|          | 0/172 [00:00<?, ?it/s]

   Epoch 25/100 | Train Loss: 0.0714 | Val Loss: 0.6707 | Train Acc: 97.19% | Val EER: 14.58% | Val Acc: 85.42%
   >>> Early stopping at epoch 25

   Using best epoch weights for final test evaluation

========== FINAL TEST RESULTS ==========
  EER          : 15.83%
  AUC          : 0.9127
  THRESHOLD    : 0.1421
  ACCURACY     : 84.14%
  PRECISION    : 80.95%
  RECALL       : 84.11%
  F1           : 82.50%

                  ABLATION A — DenseNet-121 + CBAM (Classification) | BHSig-Hindi                   
Split      Train    Val      Test          EER   Accuracy      AUC       F1    Time(s)
----------------------------------------------------------------------------------------------------
60:20:20   96       32       32         0.1583     0.8414   0.9127   0.8250     291.81

 > Results saved → /home/lawrence/workspace/thesis/thesis/checkpoints/ablation_splits/ablation_A_bhsig_hindi_results.json


                                ALL DATASETS COMPLETED SUCCESSFULLY                    